# สัปดาห์ 4 · แล็บ: จาก noise สู่ภาพ — เทรน VAE, GAN และ Diffusion ด้วยมือเราเอง

**เส้นด้ายของวัน:** โมเดลสร้างภาพ = ฟังก์ชันที่เรียน "หน้าตาของภาพที่เป็นไปได้" แล้วสุ่มภาพใหม่ออกมา
สามโมเดลวันนี้ต่างกันแค่ **วิธีสุ่ม**: VAE กระโดดจาก latent ทีเดียว · GAN ก็กระโดดทีเดียวแต่มีกรรมการคอยจับผิด · Diffusion เดินจาก noise ทีละก้าว

ชุดข้อมูลเดียวกันทั้งวัน: **Sprites 16×16** (ตัวละครเกมพิกเซล 10,000 ภาพ 5 กลุ่ม) ฝังอยู่ใน notebook นี้แล้ว ไม่ต้องอัปโหลดอะไร

| แล็บ | ทำอะไร | สิ่งที่จะได้เห็น |
|---|---|---|
| 0 | เตรียมเครื่องมือ + ดูข้อมูล | ภาพ sprite และ 5 กลุ่ม |
| 1 | VAE | latent space, interpolation, ภาพสุ่มที่ "เบลอ" |
| 2 | GAN | loss ที่แกว่ง, ภาพคมขึ้น, ความหลากหลาย |
| 3 | DDPM / DDIM | forward noising, เส้นทาง denoising, 1000 ก้าว vs 50 ก้าว |
| 4 | Diffusion แบบมีเงื่อนไข | สั่งให้สร้าง "อาหาร" หรือ "ฮีโร่" ได้ (สะพานสู่สัปดาห์หน้า) |

> **วิธีใช้:** Runtime → Change runtime type → **T4 GPU** แล้วรันทีละเซลล์จากบนลงล่าง ทุกเซลล์มีหัวข้อ "ทำอะไร / ควรเห็นอะไร / อ่านผลอย่างไร" · ทุกการสุ่มกำหนด seed ไว้ ผลจึงใกล้เคียงเดิมเมื่อรันซ้ำ (บน GPU อาจต่างเล็กน้อย เพราะการคำนวณบางอย่างไม่ deterministic)

## แล็บ 0 · เตรียมเครื่องมือและดูข้อมูล

### 0.1 ติดตั้งไลบรารี
**ทำอะไร:** ติดตั้ง `diffusers` (ไลบรารีของ Hugging Face สำหรับ diffusion) — PyTorch มีใน Colab อยู่แล้ว · ซ่อน warning ที่ไม่เกี่ยว · ตั้งค่า seed ให้รันซ้ำได้ผลเดิม
**ควรเห็นอะไร:** บรรทัดสุดท้ายพิมพ์ชื่อ GPU (`Tesla T4`) ถ้าขึ้น `cpu` ให้ไปเปลี่ยน runtime ก่อน

In [ ]:
!pip install -q -U "diffusers>=0.30" 2>/dev/null

import os, io, time, math, base64, zlib, warnings, random
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
from PIL import Image
warnings.filterwarnings("ignore")

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(42)

# ฟอนต์ไทยสำหรับกราฟ (ถ้าโหลดไม่ได้ก็ใช้ฟอนต์เริ่มต้น ไม่ล้ม)
try:
    if not os.path.exists("Sarabun-Regular.ttf"):
        import urllib.request
        urllib.request.urlretrieve("https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf", "Sarabun-Regular.ttf")
    fm.fontManager.addfont("Sarabun-Regular.ttf"); plt.rcParams["font.family"] = ["Sarabun", "DejaVu Sans"]   # DejaVu = สำรองสำหรับสัญลักษณ์
except Exception as e:
    # สำรอง: หาฟอนต์ไทยที่มีในเครื่อง (Sarabun / Loma / Garuda / Tlwg)
    thai = [f for f in fm.findSystemFonts() if any(k in os.path.basename(f) for k in ["Sarabun", "Loma", "Garuda", "Tlwg"])]
    if thai:
        fm.fontManager.addfont(thai[0]); plt.rcParams["font.family"] = [fm.FontProperties(fname=thai[0]).get_name(), "DejaVu Sans"]
    else:
        print("ใช้ฟอนต์เริ่มต้น (ไม่พบฟอนต์ไทย):", e)
plt.rcParams["figure.dpi"] = 110; plt.rcParams["axes.unicode_minus"] = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("อุปกรณ์:", torch.cuda.get_device_name(0) if device == "cuda" else "cpu")

### 0.2 โหลดชุดข้อมูล Sprites (ฝังในเซลล์นี้ — กด ▶ ได้เลย)
**ทำอะไร:** ถอดรหัสภาพ sprite 10,000 ภาพ (16×16 พิกเซล สี RGB) และป้ายกลุ่ม 5 กลุ่ม ออกจากข้อความ base64 ที่ซ่อนอยู่ในเซลล์
**ควรเห็นอะไร:** `X: (10000, 16, 16, 3)` และ `Y: (10000,)` · จำนวนภาพต่อกลุ่ม = 2,000 เท่ากันทุกกลุ่ม

In [ ]:
#@title ▶ โหลดชุดข้อมูล Sprites (ข้อมูลฝังอยู่ในเซลล์นี้) { display-mode: "form" }
_SHEET_B64 = "..."
_LABELS_B64 = "eJxFmglyG0EMA4fH/9+cRTflVFKJLWnn4AECpKq3X03P65nqnnr73vT3z873ysublf/2ezUvd9fu5pH9/szu9+D3X31/vk99j1V9v/dOzX4Pf79ss07loe/j3xLf6/n9Zc03j+2zXxZnrc0LWex7IKtsfW98L39/88N8J6pqjvkd77vDd5TO9lnxezhn+X7/nn/f9nn4W/9b71vje7KzZg6w7PDdK0tkufz8PTzZYHKw5vgdy3wLfhf5dp2XQ+WvV98syBH3+WiM8vLJigWy5/doTv+ZeLPyd8QYqjxnDNFes7JoxUixyuTD3wpsnYPmo3wmd2oug7M223xG7ayYd77dY5XHKdYVsSUb5mMxkhvGdt/HPjux1mey7DXeKm78Ft5sHQvnx1zwezmrJQyyWN4iGipB8S3AZb6//RkUBxMzuuX79/N059kdrPjt/7JgzlnG33fpzzcJkE4g7HC1GZwaW312Gi6e63CNvIp96unSWP1bOVGQo8X/MQ2hE/fn1zicLXLfxOBwt++JwZvYjMWLc8ewOalPdYzxLfMI+izFGhwraZOHY5LPOh3Xbx6vhGOfL8ibJmJijXgriUYEvl7vu6RY/E2QxltJyMEO3SZHTDjJCI5Hgmb3JHVc6w2bFIrHE+zEcB7MibNNP3Ij27TR/UiBvFBGSIxh4uTmuQm7fH+yleGZZPv2abO4ydgh5zQ8zrjw1TjPqAquVIyTeyRSO+mbdcmC76mnL5OQRDlpVrggriOpFhuAVORsQinxI4QYnmX85Ij5gfx9XBhT1uU6769hWYmlbE3IBQ0rwT2ETy+2Ih6SjNVgXDCAWF9jg3RKHGDlIc6CFRgzdyVmvQf2550HODc+CKIJhPFEfN8EyRAIxEIwLGjwyFYyYXOM5JHXzY7JPpNviMic/wkABca11y2wAEd8qyY4RZC+iCb/L/MWV8d8j2yK6R84GEgvt8rRrDsYs5Jb+Ws8ctdhw7Wu5DPAoDnHO08Qzxk/i2ETihYfH8pATnv2pZzkCAnWJZ5iMnIxx/zFB6nYrhvXk0C5ENFF6OLL2A2L5lBZkWijyC2lA7BKaQO/lugIFCYkY/oxrAFI7AfyNJhR1IxD7NjZgEnKmiqP/FmcQewFlgOsifgnCg7BTPFp8JGISKzO8/4FZpH3cXhWX4D8UTGTTTleSmL8lHUe4LSUby5BDga6cNe0V3+Uw6uFeAEkwrRxUOArN0hoUiGp9zIDIpDanaNTZ5NEj/pC8bXICeTG9YhhlIiSxqS85tU8WiUtIdyJPsAoid5mGeVziKk1o/QKJECGU9IksB0qkY8RKnX5qQsuJkgaTpMXgGCs+Vr8CjwTfHCGYqnlwXyKikGwjEXZe5VhxSGz4BR2DewQ/FkkW8W6sCpce0U87ifFxZPlw6yGiwnSBreoOwFa7/OGKCB7LXwtOwLuBSVYX10lA/DIIByIhSifFJjQhsEMICjoORz6KhWlHlCU/IzAJmY1sKXjCddEGH7gAFzCdwb8K46/2HogbQk3uR9c5el5KsuClgNReNRrjZZYwimE0pIAOSiBAiMdLtY8TvyRwdgqsEutXwhTmYBAJOw5Z1xK87EwIy92XXliEzpAmGBCMTSWYVPUmoMrTNHEjdkMUyI6ty8wuVOT+YkRGGKytYE4S54k8I1EO29TvQy6IAGwA8rDM696AHtLes+vvktwLedlRFA5A8tZX4oA2bGig5FSL6CXY73j6I8geBByql2wKyfCoWAKQIZtF6MnfCjtI5GC2AhSQEvNQYM6AXw9HExyExNwClSEhEfDyPkh+BYvspGw3TFHhypIJgzbtWwPPC1K7HBZgzdGQXfgt0A0VJ8rwkIf1J6CD7tfWDT4XC2vFmOhvxIUYKao52LwJQ7UILQc+k2RGXLUOmnVf+WZn1VQlx1BBECaqgR94pwD3yQyE8cy6LivVIwwuVxzETIzHjboMYfMckYALIkdC3F2oHsNTO6jemjInAmFEhsVQQMTRqAeo+6LLCSiZBimqqkTB0UV4Lj7hBcOxr4BpqdKJFnYv4kuChA1QsFIHUacxo5WO+uL2gNQUfwmjLHlyuSBaGSt7EVDQZku8pBiq/oZaRGFQuBEEz71u0q2jN/WLgAEOAcGPjVm7+X+ylzUsGSZKomMI++5LESYsMfNOUO2i6v9VOxfHn7NZEy2hihF/R21xA4WO8niUN9zuafYlnlhdD7OOcFKyzdcj+SDRynNGj44QvH9rDyBsKzxTl53SW2kIXW2WFUAK6BxlGDw1FGqKZKICNJ9FOFGMadX1a1kIElsrCE54Z3wT2j3wAc0OBeERRUSlhQ4XJ5z0LtqbClJrKDTpLGNmCHg1ZkQNuICQ4FIfWRdyG0L6EELd3rw9YckIWihIyz4pLbgpY0JMAdcezC9hyYYBeGayTV2meiKsO8TiJvCvbrf/44Z2gU5nlHgTqn1WZC4POEGj1eWxB7EN4yNIg+vpTzJshE1iAUx7kd2jjZLodq+ECoRgQ2huG7Ugw2rNwyrnC/WHzHtP/kh3sIRFtEC24FC9PXHHtKdtYsyOQp+20lDf8DWkRxVVfFO/1FveNr8eVS4sCwko5WyVApP1IZ+2Z2QO+gy2K5aFpN6JdwTxCxtAwSRl3YLW6PDAjSPkkvUhLhRjnXAk5yiWHo9Wo5C2EBQ7O5xQ1pBiVU7k09W8ORQpbaivYcEoTOzXGHh7XuUnQgDGoDuKjOn0TPIThGWQxJiyvLV7r5YknQqAgez3mVlm5ZlQbLvYUHiObkT7rDBh9PEKCNRrojKBMBPUl3bJ64F+fR53yEMRhJy4V/Uq18vkg/TK4QA4wWLBLljUMyVa5kBdevhK3R2XwslsvEHALImjG1FXdkf0u2pjEgfBTYWbSrMc1VppYyS2vHsgzWU2YZvHwkpsNDWXEOQ2INftRWw3gQvxRgYw0pAogtArlJSxvZjznGt4qahhMRGGWc76hO9NqB/rulSp6mIc/KCjhTMAircbeGAMKFPHjxroIFS0qqfwMOH9EYX6dOjAMWWRB5+JZxDdlfVP9ejwKA0UI71gjnWu7H1KAEHj/5uQQDGVurOq5cydcJf6oFCVaLRh7EniR4HwwHCI/70jO1s2JalLmRR+LUdYfCMzLI7j16kpKMJ8Mn1c62oJgUcEAZz9AvZ3MYCYpfHka+hYCDnM37rV75RVQ+a+yOMYrcTCHmO+EIjoq0tR13LbqndJfs7KETetUUJ4YOCYLguxXw+bJ+bycUaVtCDvqa0dA6P2H+lXeXFpJ1SEqcml+7ksFROa4o178TS0SD7ppRYyAxJslZ4iNA1tpSuJUavrULg1NpQY7sbxWXo3G6HgWUPy/tREBrxLsE1lkv9K/0WSuHR9lD/+oXYrvfoEaCo8MBfDlTqN9sZ2bhwpiJZC8Uidxn30EilijtoIFr2EgyjgoWOct6TOJ4A48PcCxTJpWi5jQMxygacVeHZuk2KprAqK9c1MlpIgLxSMDUjtXhPr5CXK5HRsWCKjRWo1fsNCGZ/bbzr2EMD/Jd+sk/9XQNPVd/4q5wqDW1XuvNoWFriFDMPhDLKUZgWIa/moM4subHdOPW6OQbgfNSArkM7KOoTZsLc2P378RXEjlMezgAxoI+yDkGIKFvSWKRVo/Mb85GOsQButj34kCmWOyj2XruVlshVIqil/Tr21qLg/1wrYRxa9qgBZQg/WWmDhxpK+6Rso8OfIcDHiMtZ4B4xen2quiHRZCeaY+0Fn6h9N6yx48EUA635PNJRM056PNWon+vPt715xmPPYnlkdQRMmkEjNAqUIKiC1nJm75RGH1lI98fmuy0wMoQCq8TZXzQBv2N/DQr9/vr+dQqbkrLKfSzr+uOgqC2d9hvX+SZdQQkKlRANJ51cswfgbyn4NUedYCg32jEYBMSGFjMNSjIwtJI1Yv7pVIQwEo0olJuXuYng86B2U8wI58rQVuqzjBWEbN0CB3qnJVR9yOijn9dlL5si3gX5xTyOJBg7nSXG/GiZzRDAGmaNYreDdD3eo4Pk9NwYjwBQC8l1+6/aOC9B0Y2TYxMUgCtTBQvYXLTvxuhlbmgEXouq49TMaSl0Yx3prQOfJkcpXdCUdhLwn9WN3Xzb/KIcNZyOG+2wWae3Unea0jZIbpZFZ8/5OkaFC+51gOaQEZLNKNF29fym7nvd4ljWkYfzb3iLyIjQWUozLqN2PLviNHNb3mYYE6rqhTHeYqy2F3TjmT5LPlF8bFBy0O2TpL8+hmqkleOjXKTH58gT8kuTBZrlDJRBjz1Vc7RvqCTdJFL4wshNxfuw+EThs/AfpS/L5TjeAvvqx9lFQ15mX1ud9/URFInDdeSCc1M6ZXY0mBX69RAIFtjFgYk6FJU2dWB7uWdho046VPq1HSGltA5HN9GjU/TdPNVvlpiB43TRkq1jbyjain2nwiV7K3lHybc07tSJcRa9OR8U5Vc8aeb85jIIj3ezb94bQ7GvR7nO+GkVKW5oXhhODgO9d+8h7fZxW9oxtgVXjrDXkHTuyPa7KrPndyrG7vvzOzDr0Fpku7YHfSVouG4fwh0dIFG6GaKbgk7XeX5UWnCZLyE5eaGeXdeF1gUimK9trMN82BopB6wQ9zIYiTWOth01Et05YuHXFigHlL0bF/5GBNcWqruaZYRedv2+4ECNnT8GR+bXqQbvPL5K18yi5+St0PLvdAqKlxmSBWvv2yPveoSOuxj4qWrlP0AWSxxSjEV8LBbE5++7HlDCUd2MXz1R8WB6J3A3tRv7q+h4YktGagvbJiZk2KH/9G9AzvTNXu6KKc77V0SC149lzipDDbKx48jXr+Bgd3qgFxfUp+u63oRVHPC7IX6FBkWAzkb+KrJIGUODXoiljpqvzPqe/wegZU4h"
_sheet = np.array(Image.open(io.BytesIO(base64.b64decode(_SHEET_B64))).convert("RGB"))     # (1600,1600,3) = 100×100 sprites
X = _sheet.reshape(100, 16, 100, 16, 3).transpose(0, 2, 1, 3, 4).reshape(10000, 16, 16, 3)  # (N,16,16,3) uint8 0–255
Y = np.frombuffer(zlib.decompress(base64.b64decode(_LABELS_B64)), dtype=np.uint8).astype(np.int64)
CLASS_NAMES = ["ฮีโร่", "ตัวประกอบ", "อาหาร", "คาถา", "หันข้าง"]   # hero / non-hero / food / spell / side-facing
print("X:", X.shape, X.dtype, "| Y:", Y.shape, "| ภาพต่อกลุ่ม:", np.bincount(Y))

### 0.3 ดูข้อมูลและเตรียม tensor
**ทำอะไร:** แสดง sprite 40 ภาพพร้อมชื่อกลุ่ม · แปลงภาพเป็น tensor รูปร่าง `(N, 3, 16, 16)` ค่าช่วง **−1 ถึง 1** (โมเดลสร้างภาพนิยมช่วงนี้ เพราะปลายทางใช้ `tanh`)
**ควรเห็นอะไร:** ตารางภาพ 5×8 ตัวละครเกมหลากสี · `x_all` ช่วงค่า −1.0 ถึง 1.0
**อ่านผลอย่างไร:** สังเกตว่าภาพกลุ่มเดียวกันมีท่าทางคล้ายกัน (ฮีโร่ยืนตรง, อาหารเป็นก้อนกลม ๆ) — นี่คือ "โครงสร้าง" ที่โมเดลต้องเรียนรู้

In [ ]:
def to_img(t):
    """tensor (N,3,16,16) ช่วง −1..1  →  numpy (N,16,16,3) ช่วง 0..1 สำหรับ imshow"""
    return ((t.detach().cpu().float().clamp(-1, 1) + 1) / 2).permute(0, 2, 3, 1).numpy()

def show_grid(imgs, titles=None, nrow=8, scale=1.0, suptitle=None):
    """imgs: numpy (N,16,16,3) 0..1 หรือ tensor (N,3,16,16) −1..1"""
    if torch.is_tensor(imgs): imgs = to_img(imgs)
    n = len(imgs); ncol = min(nrow, n); nr = math.ceil(n / ncol)
    fig, axes = plt.subplots(nr, ncol, figsize=(ncol * 1.1 * scale, nr * 1.25 * scale))
    for i, ax in enumerate(np.array(axes).reshape(-1)):
        ax.axis("off")
        if i < n:
            ax.imshow(imgs[i], interpolation="nearest")
            if titles is not None: ax.set_title(str(titles[i]), fontsize=8)
    if suptitle: fig.suptitle(suptitle, fontsize=11)
    plt.tight_layout(); plt.show()

show_grid(X[:40] / 255.0, titles=[CLASS_NAMES[c] for c in Y[:40]], nrow=8)

# (N,16,16,3) uint8 → (N,3,16,16) float ช่วง −1..1
x_all = torch.from_numpy(X).permute(0, 3, 1, 2).float() / 127.5 - 1.0
y_all = torch.from_numpy(Y)
print("x_all:", tuple(x_all.shape), "ช่วงค่า", x_all.min().item(), "ถึง", x_all.max().item())

BATCH = 128
dataset = torch.utils.data.TensorDataset(x_all, y_all)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH, shuffle=True, drop_last=True)
print("จำนวน batch ต่อ epoch:", len(loader))

## แล็บ 1 · VAE — บีบภาพเป็นรหัสสั้น ๆ แล้วคลายกลับ

**ภาพรวม (จากช่วงเช้า):** encoder บีบภาพ 16×16×3 = 768 ตัวเลข ให้เหลือรหัส `z` แค่ 16 ตัวเลข · decoder คลาย `z` กลับเป็นภาพ
สิ่งที่ทำให้ "autoencoder" กลายเป็น "**variational** autoencoder" คือ encoder ไม่ได้ให้จุดเดียว แต่ให้ **ค่าเฉลี่ย (mu) และความกว้าง (logvar)** ของบริเวณที่ภาพนี้อยู่ แล้วเราสุ่ม `z` จากบริเวณนั้น + บังคับให้บริเวณทั้งหมดรวมกันหน้าตาเหมือน Gaussian มาตรฐาน (เทอม KL) → ทำให้สุ่ม `z` ใหม่จาก N(0, I) แล้ว decode ได้ภาพที่สมเหตุสมผล

### 1.1 สร้างโมเดล
**ทำอะไร:** encoder = CNN 2 ชั้น (เหมือนสัปดาห์ 1 แต่ไม่มีชั้น classifier) → fc ออก `mu`, `logvar` · decoder = กลับทิศด้วย `ConvTranspose2d`
**ควรเห็นอะไร:** ทดสอบผ่าน 1 batch แล้วได้ `recon` รูปร่างเท่ากับภาพเข้า `(128, 3, 16, 16)` และ `z` รูปร่าง `(128, 16)`

In [ ]:
Z_DIM = 16   # ขนาดรหัส latent

class VAE(nn.Module):
    def __init__(self, z_dim=Z_DIM):
        super().__init__()
        # encoder: 16×16 → 8×8 → 4×4 → 64×4×4 = 1024 ตัวเลข
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(), nn.Flatten())
        self.fc_mu = nn.Linear(1024, z_dim)        # จุดกลางของบริเวณ
        self.fc_logvar = nn.Linear(1024, z_dim)    # ความกว้าง (log sigma²)
        # decoder: z → 4×4 → 8×8 → 16×16
        self.dec_fc = nn.Linear(z_dim, 1024)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Tanh())

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        # z = mu + sigma × ε  (สุ่มแบบที่ยัง backprop ผ่าน mu, sigma ได้)
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

    def decode(self, z):
        return self.dec(self.dec_fc(z).view(-1, 64, 4, 4))

    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar, z

vae = VAE().to(device)
xb, yb = next(iter(loader)); xb = xb.to(device)
recon, mu, logvar, z = vae(xb)
print("ภาพเข้า", tuple(xb.shape), "→ z", tuple(z.shape), "→ ภาพออก", tuple(recon.shape))

### 1.2 เทรน VAE
**ทำอะไร:** loss = **ค่าผิดพลาดของการคลายภาพ** (MSE ระหว่างภาพเข้ากับภาพออก) + β × **KL** (บังคับให้ latent เป็นระเบียบ) · เทรน 15 epoch (~1 นาทีบน T4)
**ควรเห็นอะไร:** loss ลดลง**เรียบ ๆ** ทุก epoch (เหมือน classifier สัปดาห์ 1) · กราฟ 2 เส้น: recon ลดเร็ว, KL ค่อย ๆ นิ่ง
**อ่านผลอย่างไร:** VAE เป็นโมเดลสร้างภาพที่ "เทรนง่ายที่สุด" ในสามตัววันนี้ — ไม่มีอะไรแกว่ง เพราะมี loss ตัวเดียวที่ชัดเจน

In [ ]:
def vae_loss(recon, x, mu, logvar, beta=1.0):
    # ทั้งสองเทอมคิดเป็น "ต่อภาพ"
    recon_err = F.mse_loss(recon, x, reduction="sum") / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon_err + beta * kl, recon_err, kl

VAE_EPOCHS = 15
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
hist = {"recon": [], "kl": []}
set_seed(0); t0 = time.time()
for ep in range(VAE_EPOCHS):
    vae.train(); s_r = s_k = 0.0
    for xb, _ in loader:
        xb = xb.to(device)
        recon, mu, logvar, _ = vae(xb)
        loss, r, k = vae_loss(recon, xb, mu, logvar, beta=1.0)
        opt.zero_grad(); loss.backward(); opt.step()
        s_r += r.item(); s_k += k.item()
    hist["recon"].append(s_r / len(loader)); hist["kl"].append(s_k / len(loader))
    if ep % 3 == 0 or ep == VAE_EPOCHS - 1:
        print(f"epoch {ep+1:2d}/{VAE_EPOCHS}  recon {hist['recon'][-1]:7.2f}"
              f"  KL {hist['kl'][-1]:6.2f}  ({time.time()-t0:.0f} s)")

plt.figure(figsize=(6, 3)); plt.plot(hist["recon"], label="recon (MSE)"); plt.plot(hist["kl"], label="KL")
plt.xlabel("epoch"); plt.legend(); plt.title("VAE: loss ลดลงเรียบ ๆ"); plt.grid(alpha=.3); plt.show()

### 1.3 ดูผล: คลายภาพ · สุ่มภาพใหม่ · เดินใน latent
**ทำอะไร:** (ก) ภาพจริง vs ภาพที่ VAE คลายกลับ · (ข) สุ่ม `z ~ N(0, I)` 32 ตัวแล้ว decode = **สร้างภาพใหม่** · (ค) เดินเป็นเส้นตรงใน latent ระหว่าง sprite 2 ตัว (interpolation)
**ควรเห็นอะไร:** (ก) ภาพคลายกลับจำโครงได้แต่ **เบลอ** · (ข) ภาพสุ่มดูเป็นตัวละครแต่ขอบไม่คม · (ค) ภาพค่อย ๆ "morph" จากตัวหนึ่งเป็นอีกตัว
**อ่านผลอย่างไร:** เบลอเพราะ MSE ให้รางวัล "ค่าเฉลี่ย" — เมื่อโมเดลไม่แน่ใจว่าพิกเซลควรเป็นสีไหน การตอบค่ากลาง ๆ ปลอดภัยที่สุด นี่คือจุดอ่อนที่ GAN และ Diffusion จะแก้

In [ ]:
vae.eval()
with torch.no_grad():
    xb = x_all[:16].to(device)
    recon, mu, logvar, _ = vae(xb)
    show_grid(torch.cat([xb, recon]), nrow=16, suptitle="แถวบน = ภาพจริง · แถวล่าง = VAE คลายกลับ (เบลอ)")

    set_seed(1)
    z_new = torch.randn(32, Z_DIM, device=device)                    # สุ่มรหัสใหม่จาก N(0, I)
    show_grid(vae.decode(z_new), nrow=16, suptitle="VAE สร้างภาพใหม่จาก z ที่สุ่ม")

    mu_a, _ = vae.encode(x_all[0:1].to(device)); mu_b, _ = vae.encode(x_all[7:8].to(device))
    alphas = torch.linspace(0, 1, 10, device=device).view(-1, 1)
    z_path = (1 - alphas) * mu_a + alphas * mu_b                     # เดินเส้นตรงจาก a ไป b
    show_grid(vae.decode(z_path), nrow=10, suptitle="เดินใน latent: จาก sprite #0 ไป sprite #7")

### 1.4 แผนที่ latent
**ทำอะไร:** encode ภาพทั้ง 10,000 ได้ `mu` 16 มิติ → ฉายลง 2 มิติด้วย PCA แล้วระบายสีตามกลุ่ม
**ควรเห็นอะไร:** จุดสีเดียวกันเกาะกลุ่มกัน (อาหารอยู่โซนหนึ่ง ฮีโร่อีกโซน) แม้ VAE **ไม่เคยเห็นป้ายกลุ่มเลย**
**อ่านผลอย่างไร:** latent space คือ "แผนที่ของภาพ" — ตำแหน่งใกล้กัน = ภาพคล้ายกัน สัปดาห์หน้า Stable Diffusion จะทำงานบนแผนที่แบบนี้ (แต่ใหญ่กว่า) แทนที่จะทำบนพิกเซลตรง ๆ

In [ ]:
with torch.no_grad():
    MU = torch.cat([vae.encode(x_all[i:i + 512].to(device))[0].cpu() for i in range(0, len(x_all), 512)])
# PCA แบบง่าย: หาแกนหลัก 2 แกนด้วย SVD
MUc = MU - MU.mean(0); U, S, V = torch.linalg.svd(MUc, full_matrices=False); P2 = (MUc @ V[:2].T).numpy()
plt.figure(figsize=(5.5, 4.5))
for c in range(5):
    m = Y == c; plt.scatter(P2[m, 0][:800], P2[m, 1][:800], s=4, alpha=.5, label=CLASS_NAMES[c])
plt.legend(markerscale=3); plt.title("latent ของ VAE (ฉาย 16 → 2 มิติ)"); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

## แล็บ 2 · GAN — เกมสองฝ่าย: คนปลอม vs กรรมการ

**ภาพรวม (จากช่วงเช้า):** **Generator (G)** รับ noise `z` แล้วผลิตภาพ · **Discriminator (D)** รับภาพแล้วตอบว่า "จริง" หรือ "ปลอม" · เทรนสลับกัน: D ฝึกจับผิด, G ฝึกหลอก D
ไม่มี MSE กับภาพจริงเลย → G ไม่ถูกบังคับให้ตอบ "ค่าเฉลี่ย" → ภาพ**คม**กว่า VAE แต่แลกมาด้วยการเทรนที่**แกว่ง**

### 2.1 สร้าง G และ D (DCGAN ขนาดเล็ก)
**ทำอะไร:** G มีโครงเหมือน decoder ของ VAE (z → 4×4 → 8×8 → 16×16) · D มีโครงเหมือน encoder ของ VAE แต่ออกค่าเดียว (logit "จริงไหม")
**ควรเห็นอะไร:** G(z) ได้ `(128, 3, 16, 16)` · D(ภาพ) ได้ `(128, 1)`

In [ ]:
NZ = 64   # ขนาด noise ที่ป้อน G

class Generator(nn.Module):
    def __init__(self, nz=NZ):
        super().__init__()
        self.fc = nn.Linear(nz, 256 * 4 * 4)
        self.net = nn.Sequential(
            nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),  # 4→8
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),    # 8→16
            nn.Conv2d(64, 3, 3, 1, 1), nn.Tanh())
    def forward(self, z): return self.net(self.fc(z).view(-1, 256, 4, 4))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2),                          # 16→8
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),   # 8→4
            nn.Flatten(), nn.Linear(128 * 4 * 4, 1))     # 1 logit: บวก = จริง, ลบ = ปลอม
    def forward(self, x): return self.net(x)

G, D = Generator().to(device), Discriminator().to(device)
z = torch.randn(BATCH, NZ, device=device)
print("G(z):", tuple(G(z).shape), "| D(ภาพ):", tuple(D(G(z)).shape))

### 2.2 เทรน GAN
**ทำอะไร:** ทุก batch: (1) อัปเดต D ให้ตอบ "จริง" กับภาพจริง และ "ปลอม" กับ G(z) · (2) อัปเดต G ให้ D ตอบ "จริง" กับ G(z) · เทรน 25 epoch (~2 นาทีบน T4) · เก็บภาพจาก noise ชุดเดิม (`z_fixed`) ทุก 5 epoch
**ควรเห็นอะไร:** loss ของ G และ D **แกว่งไปมา ไม่ลดลงเรียบ ๆ** แบบ VAE · ภาพจาก `z_fixed` ค่อย ๆ จาก noise กลายเป็นตัวละคร
**อ่านผลอย่างไร:** loss ไม่ใช่ตัววัดคุณภาพของ GAN (ถ้า D แพ้ตลอด loss ของ G ต่ำแต่ภาพอาจแย่) — ต้อง**ดูภาพ**เสมอ

In [ ]:
GAN_EPOCHS = 25
optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()
set_seed(0); z_fixed = torch.randn(16, NZ, device=device)
ghist, dhist, snapshots = [], [], []
t0 = time.time()
for ep in range(GAN_EPOCHS):
    for xb, _ in loader:
        xb = xb.to(device)
        real_lbl = torch.ones(BATCH, 1, device=device)
        fake_lbl = torch.zeros(BATCH, 1, device=device)
        # --- (1) ฝึกกรรมการ D ---
        fake = G(torch.randn(BATCH, NZ, device=device))
        lossD = bce(D(xb), real_lbl) + bce(D(fake.detach()), fake_lbl)
        optD.zero_grad(); lossD.backward(); optD.step()
        # --- (2) ฝึกคนปลอม G: อยากให้ D ตอบ "จริง" ---
        lossG = bce(D(fake), real_lbl)
        optG.zero_grad(); lossG.backward(); optG.step()
        ghist.append(lossG.item()); dhist.append(lossD.item())
    if (ep + 1) % 5 == 0 or ep == 0:
        with torch.no_grad(): snapshots.append((ep + 1, to_img(G(z_fixed))))
        print(f"epoch {ep+1:2d}/{GAN_EPOCHS}  lossD {np.mean(dhist[-len(loader):]):.3f}"
              f"  lossG {np.mean(ghist[-len(loader):]):.3f}  ({time.time()-t0:.0f} s)")

plt.figure(figsize=(6, 3)); plt.plot(dhist, label="D", alpha=.7); plt.plot(ghist, label="G", alpha=.7)
plt.xlabel("step"); plt.legend(); plt.title("GAN: loss แกว่ง ไม่ลดเรียบ ๆ"); plt.grid(alpha=.3); plt.show()
for ep, imgs in snapshots:
    show_grid(imgs, nrow=16, scale=0.8, suptitle=f"G(z_fixed) หลัง epoch {ep}")

### 2.3 เทียบ VAE กับ GAN บนโจทย์เดียวกัน + วัดความหลากหลาย
**ทำอะไร:** สุ่มภาพ 32 ใบจาก VAE และจาก GAN · วัด **ความหลากหลาย** = ระยะเฉลี่ยระหว่างภาพที่สุ่มมาเป็นคู่ ๆ (ยิ่งต่ำ = ภาพยิ่งซ้ำกัน) เทียบกับระยะในชุดข้อมูลจริง
**ควรเห็นอะไร:** GAN คมกว่า VAE ชัดเจน · ตัวเลขความหลากหลาย: ข้อมูลจริง ≈ 25 · VAE **ต่ำกว่า** (ภาพเบลอ สีจาง → ภาพต่างกันน้อยลง) · GAN ถ้าเทรนดีจะ**ใกล้เคียงข้อมูลจริง** แต่ถ้าเกิด mode collapse จะ**ตกฮวบ** (ต่ำกว่า 10)
**อ่านผลอย่างไร:** ตัวชี้วัดนี้จับได้ 2 อย่าง: ภาพซ้ำกัน (mode collapse: G เจอ "ท่าไม้ตาย" ที่หลอก D ได้แล้วผลิตแต่แบบนั้น) และภาพเบลอ (VAE) · GAN รอบนี้อาจไม่ collapse — ลองรันเซลล์ 2.2 ซ้ำอีก 25 epoch แล้ววัดใหม่ (หลาย ๆ รอบมักเริ่มซ้ำ) นี่คือความ "ไม่แน่นอน" ของ GAN ที่ Diffusion จะแก้

In [ ]:
def diversity(imgs, n_pairs=2000, seed=0):
    """ระยะ L2 เฉลี่ยระหว่างภาพคู่สุ่ม (ภาพ tensor −1..1 รูปร่าง (N,3,16,16))"""
    g = torch.Generator().manual_seed(seed); flat = imgs.detach().cpu().flatten(1)
    i = torch.randint(0, len(flat), (n_pairs,), generator=g)
    j = torch.randint(0, len(flat), (n_pairs,), generator=g)
    return (flat[i] - flat[j]).norm(dim=1).mean().item()

G.eval(); vae.eval()
with torch.no_grad():
    set_seed(1)
    img_vae = vae.decode(torch.randn(256, Z_DIM, device=device))
    img_gan = G(torch.randn(256, NZ, device=device))
show_grid(img_vae[:32], nrow=16, scale=0.8, suptitle="VAE สุ่ม 32 ภาพ")
show_grid(img_gan[:32], nrow=16, scale=0.8, suptitle="GAN สุ่ม 32 ภาพ")
print(f"ความหลากหลาย (ยิ่งสูงยิ่งหลากหลาย):  ข้อมูลจริง {diversity(x_all[:256]):.2f}   VAE {diversity(img_vae):.2f}   GAN {diversity(img_gan):.2f}")

## แล็บ 3 · Diffusion (DDPM) — เดินจาก noise ทีละก้าว

**ภาพรวม (จากช่วงเช้า):**
- **Forward (เติม noise):** ภาพจริง `x0` → เติม noise ทีละนิด 1,000 ก้าว จนเป็น noise ล้วน — ขั้นนี้ไม่ต้องเรียน เป็นสูตรตายตัว
- **Reverse (ลบ noise):** เครือข่ายรับภาพที่มี noise + หมายเลขก้าว `t` แล้ว **ทายว่า noise ที่ถูกเติมเข้ามาคืออะไร** → เอาออกทีละนิดจนได้ภาพ
- **เทรน:** สุ่มภาพ, สุ่ม `t`, สุ่ม noise → เติม → ให้เครือข่ายทาย noise → loss = MSE(noise ที่ทาย, noise จริง) — **loss ตัวเดียว ไม่มีกรรมการ** เหมือน VAE แต่ภาพคมเหมือน GAN

### 3.1 Forward: ดูภาพค่อย ๆ จมหายไปใน noise
**ทำอะไร:** สร้าง `DDPMScheduler` 1,000 ก้าว (ตารางว่าแต่ละก้าวเติม noise เท่าไร) · ใช้ `add_noise` เติม noise ให้ sprite ตัวเดียวกันที่ `t = 0, 50, 100, ..., 999`
**ควรเห็นอะไร:** แถวภาพที่ค่อย ๆ จาง: t=0 ชัด, ราว t=300 ยังพอเห็นโครง, t=999 เป็น noise ล้วน · กราฟ `alphas_cumprod` (สัดส่วนของ "ภาพเดิม" ที่ยังเหลือ) ลดจาก 1 ไป 0
**อ่านผลอย่างไร:** ภาพที่ `t` ต่างกันคือ "ระดับความยาก" ต่างกัน — ที่ `t` เล็ก โมเดลแค่ลบ noise บาง ๆ, ที่ `t` ใหญ่ โมเดลต้อง "จินตนาการ" โครงภาพขึ้นมาเอง

In [ ]:
from diffusers import DDPMScheduler, DDIMScheduler, UNet2DModel

T = 1000
# ตารางเติม noise แบบ cosine (เหมาะกับภาพเล็ก)
noise_sched = DDPMScheduler(num_train_timesteps=T, beta_schedule="squaredcos_cap_v2")

x0 = x_all[3:4].repeat(11, 1, 1, 1)        # sprite ตัวเดียว 11 ก๊อปปี้
ts = torch.tensor([0, 50, 100, 200, 300, 400, 500, 600, 700, 850, 999])
set_seed(0); noise = torch.randn_like(x0)
x_t = noise_sched.add_noise(x0, noise, ts)  # x_t = √ᾱ_t·x0 + √(1−ᾱ_t)·noise
show_grid(x_t, titles=[f"t={t}" for t in ts.tolist()], nrow=11, suptitle="Forward: เติม noise ทีละก้าว")

ac = noise_sched.alphas_cumprod.numpy()
plt.figure(figsize=(6, 2.8)); plt.plot(ac, label="ᾱ_t  (สัดส่วนภาพเดิมที่เหลือ)"); plt.plot(1 - ac, label="1−ᾱ_t  (สัดส่วน noise)")
plt.xlabel("t"); plt.legend(); plt.grid(alpha=.3); plt.title("ตารางเติม noise 1,000 ก้าว"); plt.show()

### 3.2 สร้างตัวทาย noise (U-Net ขนาดเล็ก)
**ทำอะไร:** ใช้ `UNet2DModel` จาก diffusers — CNN รูปตัว U (ย่อภาพลงแล้วขยายกลับ มี skip connection) ที่รับ `t` เป็น input เพิ่ม · **ภาพเข้าและภาพออกมีขนาดเท่ากัน** เพราะงานคือ "ทาย noise ที่มีขนาดเท่าภาพ"
**ควรเห็นอะไร:** `unet(x_t, t).sample` มีรูปร่าง `(11, 3, 16, 16)` เท่ากับภาพเข้า

In [ ]:
def make_unet(num_class_embeds=None):
    return UNet2DModel(
        sample_size=16, in_channels=3, out_channels=3, layers_per_block=1,
        block_out_channels=(64, 128, 128),
        # 16 → 8 → 4 มี self-attention ที่ชั้น 8×8 และ 4×4
        down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
        num_class_embeds=num_class_embeds)   # แล็บ 4 จะใส่ป้ายกลุ่มตรงนี้

unet = make_unet().to(device)
globals().pop("ddpm_losses", None)      # สร้างโมเดลใหม่ = ล้างสถานะ "เทรนแล้ว" ของเซลล์ 3.3
with torch.no_grad():
    out = unet(x_t.to(device), ts.to(device)).sample
print("ภาพเข้า", tuple(x_t.shape), "→ noise ที่ทาย", tuple(out.shape))

### 3.3 เทรน DDPM
**ทำอะไร:** ทุก batch: สุ่ม `t` ให้แต่ละภาพ → สุ่ม noise → `add_noise` → U-Net ทาย noise → `mse_loss(ทาย, จริง)` · เทรน 30 epoch (~4 นาทีบน T4)
**ควรเห็นอะไร:** loss ลดลง**เรียบ ๆ** (เหมือน VAE ไม่เหมือน GAN) จากราว 0.4 ลงมาต่ำกว่า 0.1
**อ่านผลอย่างไร:** เทียบกับแล็บ 2 — ไม่มีสองฝ่ายสู้กัน ไม่มี mode collapse ที่ต้องกังวล แค่ "ทาย noise ให้แม่น" งานเดียว นี่คือเหตุผลที่วงการย้ายจาก GAN มา Diffusion

In [ ]:
def train_diffusion(model, epochs, cond=False, p_uncond=0.1, lr=2e-4, seed=0, tag="DDPM"):
    """ลูปเทรนตัวทาย noise · cond=True = ป้อนป้ายกลุ่มด้วย (ป้าย 5 = 'ไม่บอกกลุ่ม' ใช้สุ่มแทนด้วยความน่าจะเป็น p_uncond)"""
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * len(loader))
    losses = []; set_seed(seed); t0 = time.time(); model.train()
    for ep in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            t = torch.randint(0, T, (xb.size(0),), device=device)   # ก้าวสุ่มต่อภาพ
            noise = torch.randn_like(xb)
            x_noisy = noise_sched.add_noise(xb, noise, t)            # forward 1 บรรทัด
            if cond:   # แล็บ 4: สุ่มซ่อนป้าย (ใส่ 5 = "ไม่บอก")
                drop = torch.rand(len(yb), device=device) < p_uncond
                yb = torch.where(drop, torch.full_like(yb, 5), yb)
                pred = model(x_noisy, t, class_labels=yb).sample
            else:
                pred = model(x_noisy, t).sample
            loss = F.mse_loss(pred, noise)                           # ทาย noise ให้แม่น
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            losses.append(loss.item())
        if (ep + 1) % 5 == 0 or ep == 0:
            print(f"[{tag}] epoch {ep+1:2d}/{epochs}  loss {np.mean(losses[-len(loader):]):.4f}  ({time.time()-t0:.0f} s)")
    return losses

DDPM_EPOCHS = 30
if "ddpm_losses" in globals():
    print("เทรนไปแล้ว — ข้าม (ถ้าต้องการเทรนใหม่: รันเซลล์ 3.2 อีกครั้ง แล้วค่อยรันเซลล์นี้)")
else:
    ddpm_losses = train_diffusion(unet, DDPM_EPOCHS, tag="DDPM")
plt.figure(figsize=(6, 3)); plt.plot(ddpm_losses, alpha=.8); plt.xlabel("step"); plt.ylabel("MSE ของ noise")
plt.title("DDPM: loss ลดลงเรียบ ๆ"); plt.grid(alpha=.3); plt.show()

### 3.4 Reverse: สร้างภาพจาก noise ล้วน (DDPM 1,000 ก้าว)
**ทำอะไร:** เริ่มจาก noise สุ่ม → วนลูป `t = 999, 998, …, 0`: U-Net ทาย noise → `scheduler.step` ลบออกหนึ่งก้าว · เก็บภาพระหว่างทางทุก 100 ก้าว
**ควรเห็นอะไร:** แถวภาพที่ค่อย ๆ โผล่จาก noise: ก้าวแรก ๆ ยังเป็นสีเทา ๆ, กลางทางเริ่มเห็นโครงตัวละคร, ก้าวท้าย ๆ เก็บรายละเอียด · ตารางภาพ 32 ใบ**คม**และ**หลากหลาย** (ใกล้เคียง GAN ที่เทรนสำเร็จ แต่ไม่มีความเสี่ยง collapse) · เวลาที่ใช้ (วินาที)
**อ่านผลอย่างไร:** โครงใหญ่ (สี, ท่าทาง) ถูกตัดสินตอน `t` สูง รายละเอียดเล็กมาตอน `t` ต่ำ — สัปดาห์หน้าเราจะใช้ข้อเท็จจริงนี้ทำ img2img (เริ่มจากกลางทางแทนที่จะเริ่มจาก noise ล้วน)

In [ ]:
@torch.no_grad()
def sample(model, scheduler, n=32, steps=None, labels=None, guidance=1.0, keep_every=None, seed=0):
    """สร้างภาพจาก noise · steps=None = ใช้ทุกก้าวของ scheduler · labels/guidance ใช้ในแล็บ 4"""
    model.eval(); g = torch.Generator(device=device).manual_seed(seed)
    scheduler.set_timesteps(steps or scheduler.config.num_train_timesteps)
    x = torch.randn(n, 3, 16, 16, device=device, generator=g); frames = []
    for i, t in enumerate(scheduler.timesteps):
        if labels is None:
            eps = model(x, t).sample
        else:
            eps_c = model(x, t, class_labels=labels).sample
            if guidance == 1.0: eps = eps_c
            else:
                null = torch.full_like(labels, 5)                     # ป้าย "ไม่บอกกลุ่ม"
                eps_u = model(x, t, class_labels=null).sample
                eps = eps_u + guidance * (eps_c - eps_u)              # classifier-free guidance
        x = scheduler.step(eps, t, x, generator=g).prev_sample
        last = i == len(scheduler.timesteps) - 1
        if keep_every and (i % keep_every == 0 or last): frames.append((int(t), x[0:1].clone()))
    return x, frames

t0 = time.time()
imgs_ddpm, frames = sample(unet, noise_sched, n=32, keep_every=100)
t_ddpm = time.time() - t0
show_grid(torch.cat([f[1] for f in frames]), titles=[f"t={f[0]}" for f in frames], nrow=11, suptitle="Reverse: ภาพค่อย ๆ โผล่จาก noise")
show_grid(imgs_ddpm, nrow=16, scale=0.8, suptitle=f"DDPM 1,000 ก้าว — 32 ภาพ ใช้เวลา {t_ddpm:.1f} s")

### 3.5 DDIM: โมเดลเดิม แต่เดินก้าวใหญ่ขึ้น
**ทำอะไร:** สร้าง `DDIMScheduler` จาก config เดียวกัน (ตารางเติม noise เดิม) แล้ว sample ด้วย **50, 20 และ 10 ก้าว** โดย**ไม่ต้องเทรนใหม่** · เทียบเวลาและภาพกับ DDPM 1,000 ก้าว
**ควรเห็นอะไร:** 50 ก้าวได้ภาพใกล้เคียง 1,000 ก้าวแต่เร็วกว่าราว 20 เท่า · 10 ก้าวเริ่มเห็นภาพเบลอ/โครงหลุด · ตารางเวลา
**อ่านผลอย่างไร:** DDPM เดินแบบสุ่มทีละก้าวเล็ก ๆ · DDIM เดินแบบ "กำหนดทาง" (deterministic: ภาพขึ้นกับ noise ตั้งต้นอย่างเดียว) จึงข้ามก้าวได้ และความหลากหลายไม่ตก เพราะความหลากหลายมาจาก noise ตั้งต้น ไม่ใช่จากการสุ่มระหว่างทาง — Stable Diffusion ที่ใช้กันจริงใช้ 20–50 ก้าวแบบนี้ ไม่ใช่ 1,000

In [ ]:
ddim_sched = DDIMScheduler.from_config(noise_sched.config)
rows = [("DDPM", 1000, t_ddpm, imgs_ddpm)]
for steps in [50, 20, 10]:
    t0 = time.time()
    imgs, _ = sample(unet, ddim_sched, n=32, steps=steps)
    rows.append(("DDIM", steps, time.time() - t0, imgs))

print(f"{'วิธี':6s} {'ก้าว':>5s} {'เวลา (s)':>9s} {'ความหลากหลาย':>14s}")
for name, steps, sec, imgs in rows:
    print(f"{name:6s} {steps:5d} {sec:9.1f} {diversity(imgs):14.2f}")
for name, steps, sec, imgs in rows:
    show_grid(imgs[:16], nrow=16, scale=0.8, suptitle=f"{name} {steps} ก้าว ({sec:.1f} s)")

## แล็บ 4 · Diffusion แบบมีเงื่อนไข — สั่งได้ว่าจะเอา "อาหาร" หรือ "ฮีโร่" (สะพานสู่สัปดาห์หน้า)

**ภาพรวม:** แล็บ 3 สร้างภาพได้แต่**เลือกไม่ได้**ว่าจะได้กลุ่มไหน · วิธีแก้ง่ายมาก: ป้อน**ป้ายกลุ่ม**เข้า U-Net ด้วยตอนเทรน (`class_labels=`) · สัปดาห์หน้า Stable Diffusion ทำแบบเดียวกันเป๊ะ แค่เปลี่ยน "ป้ายกลุ่ม 5 แบบ" เป็น "ข้อความอะไรก็ได้" ที่แปลงเป็น vector ด้วย CLIP

เคล็ดลับที่ใช้จริง: ตอนเทรน **สุ่มซ่อนป้าย 10%** (ใส่ป้าย 5 = "ไม่บอก") → โมเดลตัวเดียวทายได้ทั้งแบบรู้และไม่รู้เงื่อนไข → ตอน sample เอาสองคำทายมา**ลบกันแล้วขยาย** (`guidance`) เพื่อให้ภาพตรงเงื่อนไขขึ้น = **classifier-free guidance (CFG)** ตัวเดียวกับ "guidance scale" ใน Stable Diffusion

### 4.1 เทรน U-Net แบบมีเงื่อนไข
**ทำอะไร:** สร้าง U-Net ใหม่ที่รับป้าย 6 แบบ (5 กลุ่ม + "ไม่บอก") · เทรน 30 epoch ด้วยลูปเดิม `cond=True` (~4 นาทีบน T4)
**ควรเห็นอะไร:** loss ลดลงเรียบ ๆ ใกล้เคียงแล็บ 3

In [ ]:
if "cond_losses" in globals():
    print("เทรนไปแล้ว — ข้าม (ถ้าต้องการเทรนใหม่: del cond_losses แล้วรันเซลล์นี้อีกครั้ง)")
else:
    unet_c = make_unet(num_class_embeds=6).to(device)   # 0–4 = กลุ่ม, 5 = "ไม่บอกกลุ่ม"
    cond_losses = train_diffusion(unet_c, DDPM_EPOCHS, cond=True, p_uncond=0.1, tag="cond")
plt.figure(figsize=(6, 3)); plt.plot(cond_losses, alpha=.8); plt.xlabel("step"); plt.title("Diffusion แบบมีเงื่อนไข: loss"); plt.grid(alpha=.3); plt.show()

### 4.2 สั่งสร้างทีละกลุ่ม และลองหมุน guidance
**ทำอะไร:** sample ด้วย DDIM 50 ก้าว ป้อนป้ายกลุ่ม 0–4 กลุ่มละ 8 ภาพ (guidance = 1 คือไม่ขยาย) · แล้วลองกลุ่ม "อาหาร" ที่ `guidance = 1, 3, 6` จาก noise ชุดเดียวกัน
**ควรเห็นอะไร:** แต่ละแถวได้ภาพตรงกลุ่ม (อาหาร = ก้อนกลม ๆ, ฮีโร่ = ตัวยืน) · guidance สูงขึ้น → ภาพ "ตรงกลุ่ม" ชัดขึ้น สีจัดขึ้น แต่ความหลากหลายลดลง และถ้าสูงเกินภาพจะเริ่มเพี้ยน
**อ่านผลอย่างไร:** นี่คือหน้าตาของปุ่ม **guidance scale / CFG** ใน Stable Diffusion ที่สัปดาห์หน้าจะได้หมุนกับ prompt ข้อความจริง ๆ

In [ ]:
labels = torch.arange(5, device=device).repeat_interleave(8)   # กลุ่ม 0..4 กลุ่มละ 8
imgs_c, _ = sample(unet_c, ddim_sched, n=40, steps=50, labels=labels, guidance=1.0)
show_grid(imgs_c, titles=[CLASS_NAMES[c] for c in labels.tolist()], nrow=8,
          suptitle="สั่งสร้างทีละกลุ่ม (guidance = 1)")

food = torch.full((16,), 2, device=device)   # กลุ่ม 2 = อาหาร
for w in [1.0, 3.0, 6.0]:
    imgs_w, _ = sample(unet_c, ddim_sched, n=16, steps=50, labels=food, guidance=w, seed=7)
    show_grid(imgs_w, nrow=16, scale=0.8,
              suptitle=f"'อาหาร' ที่ guidance = {w:.0f}   (ความหลากหลาย {diversity(imgs_w):.2f})")

## สรุปวันนี้ — สามโมเดล ชุดข้อมูลเดียว

| | VAE | GAN | Diffusion |
|---|---|---|---|
| สุ่มภาพอย่างไร | z → decoder ทีเดียว | z → G ทีเดียว | noise → ลบทีละก้าว (10–1,000 ก้าว) |
| loss | MSE + KL (ตัวเดียว นิ่ง) | สองฝ่ายสู้กัน (แกว่ง) | MSE ของ noise (ตัวเดียว นิ่ง) |
| ภาพ | เบลอ | คม | คม |
| ความหลากหลาย | ดี | เสี่ยง mode collapse | ดี |
| ความเร็วตอนสร้าง | เร็วมาก | เร็วมาก | ช้า (แก้ด้วย DDIM / latent) |
| ใช้ต่อที่ไหน | ตัวบีบภาพของ Stable Diffusion | ตัวช่วย decoder (VQ-GAN) | หัวใจของ Stable Diffusion |

**สัปดาห์หน้า:** เอา VAE (แล็บ 1) มาบีบภาพใหญ่ 512×512 ให้เล็กก่อน → ทำ diffusion (แล็บ 3) บน latent → ใส่เงื่อนไขเป็น**ข้อความ** (แล็บ 4) = **Stable Diffusion** · แล้วต่อด้วย img2img, ControlNet และ LoRA (ตัวเดียวกับสัปดาห์ 3)

**การบ้าน:** "3 โมเดล 1 ชุดข้อมูล" — ดูรายละเอียดในเอกสารรายวิชาและ `HW4_Starter.ipynb`